In [ ]:
import json
from sklearn.model_selection import train_test_split
import pandas as pd
import hashlib
import re
import unicodedata
from pathlib import Path


In [ ]:

df = pd.read_json(
    r"C:\Users\vishn\Downloads\resume_extraction_cleaned.jsonl",
    lines=True
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nTotal records:", len(df))
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset loaded successfully!
Shape: (4879, 3)
Columns: ['system', 'user', 'assistant']


In [26]:
expected_fields = {
    "current_title",
    "previous_titles",
    "current_company",
    "previous_companies",
    "years_experience",
    "seniority",
    "primary_domain",
    "industries",
    "core_skills",
    "secondary_skills",
    "tools",
    "leadership_experience",
    "key_achievements",
    "location",
    "summary"
}

for index, assistant in enumerate(df["assistant"]):
    assert isinstance(assistant, dict), f"Invalid output at row {index}"
    assert set(assistant.keys()) == expected_fields, f"Schema mismatch at row {index}"

print("All assistant outputs have the expected schema!")

All assistant outputs have the expected schema!


In [27]:
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [28]:
def calculate_tokens(row):

    messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
        {
            "role": "assistant",
            "content": json.dumps(
                row["assistant"],
                ensure_ascii=False
            )
        }
    ]

    encoding = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    # Extract the actual token IDs
    token_ids = encoding["input_ids"]

    # Count tokens
    return len(token_ids)

In [29]:
df["token_length"] = df.apply(
    calculate_tokens,
    axis=1
)

print(df[["token_length"]].head())
print(df["token_length"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

   token_length
0          1343
1          1489
2          1474
3          1591
4          1466
count    4879.000000
mean     1662.127075
std       973.779642
min       191.000000
50%      1420.000000
75%      1883.000000
90%      2842.400000
95%      3741.300000
99%      5330.000000
max      9389.000000
Name: token_length, dtype: float64


In [30]:
row = df.iloc[0]

messages = [
    {
        "role": "system",
        "content": row["system"]
    },
    {
        "role": "user",
        "content": row["user"]
    },
    {
        "role": "assistant",
        "content": json.dumps(
            row["assistant"],
            ensure_ascii=False
        )
    }
]

token_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print("Return type:", type(token_ids))
print("Return value type:", type(token_ids).__name__)

if isinstance(token_ids, dict):
    print("Dictionary keys:", token_ids.keys())
    print("Input IDs shape:", token_ids["input_ids"].shape)
else:
    print("Token count:", len(token_ids))
    print("First 20 tokens:", token_ids[:20])

Return type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Return value type: BatchEncoding
Token count: 2
First 20 tokens: [Encoding(num_tokens=1343, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]


In [ ]:
print(
    df["token_length"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)
longest_resumes = df.nlargest(
    10,
    "token_length"
)

print(
    longest_resumes[
        ["token_length"]
    ]
)

count    4879.000000
mean     1662.127075
std       973.779642
min       191.000000
50%      1420.000000
75%      1883.000000
90%      2842.400000
95%      3741.300000
99%      5330.000000
max      9389.000000
Name: token_length, dtype: float64


In [37]:
def calculate_token_breakdown(row):

    # System + user messages
    prompt_messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]}
    ]

    # Complete conversation
    full_messages = prompt_messages + [
        {
            "role": "assistant",
            "content": json.dumps(
                row["assistant"],
                ensure_ascii=False
            )
        }
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    return pd.Series({
        "prompt_tokens": len(prompt_ids),
        "total_tokens": len(full_ids),
        "completion_tokens": len(full_ids) - len(prompt_ids)
    })

In [38]:
limits = [2048, 3072, 4096, 6144, 8192, 10240]

for limit in limits:

    count = (df["token_length"] > limit).sum()

    percentage = count / len(df) * 100

    print(
        f"Limit: {limit:5} | "
        f"Truncated: {count:4} | "
        f"Percentage: {percentage:.2f}%"
    )

Limit:  2048 | Truncated:  999 | Percentage: 20.48%
Limit:  3072 | Truncated:  416 | Percentage: 8.53%
Limit:  4096 | Truncated:  194 | Percentage: 3.98%
Limit:  6144 | Truncated:   14 | Percentage: 0.29%
Limit:  8192 | Truncated:    2 | Percentage: 0.04%
Limit: 10240 | Truncated:    0 | Percentage: 0.00%


In [40]:
token_breakdown = df.apply(
    calculate_token_breakdown,
    axis=1
)

df = pd.concat([df, token_breakdown], axis=1)

print(df.columns.tolist())

['system', 'user', 'assistant', 'token_length', 'prompt_tokens', 'total_tokens', 'completion_tokens']


In [41]:
assert (df["token_length"] == df["total_tokens"]).all()

assert (
    df["prompt_tokens"] + df["completion_tokens"]
    == df["total_tokens"]
).all()

print("All token counts are consistent!")

All token counts are consistent!


In [ ]:
print("PROMPT TOKEN STATISTICS")
print(df["prompt_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("\nCOMPLETION TOKEN STATISTICS")
print(df["completion_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

PROMPT TOKEN STATISTICS
count    4879.000000
mean     1388.593564
std       960.106824
min        75.000000
50%      1150.000000
90%      2552.400000
95%      3429.700000
99%      5017.960000
max      9100.000000
Name: prompt_tokens, dtype: float64

COMPLETION TOKEN STATISTICS
count    4879.000000
mean      273.533511
std        33.893462
min       116.000000
50%       272.000000
90%       317.000000
95%       331.000000
99%       359.220000
max       439.000000
Name: completion_tokens, dtype: float64


In [44]:

def check_token_alignment(row):
    prompt_messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]}
    ]

    full_messages = prompt_messages + [{
        "role": "assistant",
        "content": json.dumps(row["assistant"], ensure_ascii=False)
    }]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    return full_ids[:len(prompt_ids)] == prompt_ids


alignment_results = df.apply(check_token_alignment, axis=1)

print("Aligned:", alignment_results.sum())
print("Not aligned:", (~alignment_results).sum())

Aligned: 4879
Not aligned: 0


In [45]:
alignment_results = df.apply(check_token_alignment, axis=1)

print("Aligned:", alignment_results.sum())
print("Not aligned:", (~alignment_results).sum())

Aligned: 4879
Not aligned: 0


In [ ]:

def normalize_resume(text):
    if not isinstance(text, str):
        return ""

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Normalize capitalization and whitespace
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()

    return text


def generate_hash(text):
    normalized_text = normalize_resume(text)
    return hashlib.sha256(
        normalized_text.encode("utf-8")
    ).hexdigest()


df["resume_hash"] = df["user"].apply(generate_hash)

In [47]:
total_records = len(df)

duplicate_mask = df["resume_hash"].duplicated(keep=False)

duplicate_records = duplicate_mask.sum()
unique_resumes = df["resume_hash"].nunique()
duplicate_groups = (
    df.loc[duplicate_mask, "resume_hash"].nunique()
)

print("Total records:", total_records)
print("Unique resumes:", unique_resumes)
print("Records belonging to duplicate groups:", duplicate_records)
print("Number of duplicate groups:", duplicate_groups)

Total records: 4879
Unique resumes: 4877
Records belonging to duplicate groups: 4
Number of duplicate groups: 2


In [49]:
duplicate_df = df[duplicate_mask].sort_values("resume_hash")

for resume_hash, group in duplicate_df.groupby("resume_hash"):
    print("\nDuplicate group:")
    print("Number of copies:", len(group))
    print("DataFrame indices:", group.index.tolist())
    print("Resume preview:", group.iloc[0]["user"][:300])


Duplicate group:
Number of copies: 2
DataFrame indices: [1554, 1578]
Resume preview: Resume:
FINANCE OFFICER
Professional Summary
To attain a full-time position in the accounting and finance field where I can utilize the concepts I have learned and the experience I have gained to
add value to the company. 25 years of work experience. Fields of expertise include government accounting

Duplicate group:
Number of copies: 2
DataFrame indices: [797, 863]
Resume preview: Resume:
STOREKEEPER II
Professional Summary
The purpose of this document is to secure a responsible career position that fully utilizes educational knowledge and experience while contributing
to the goals and values of this organization. Having a friendly attitude and worm personality is an apprecia


In [50]:
print("Before removing duplicates:", len(df))

df = df.drop_duplicates(
    subset="resume_hash",
    keep="first"
).copy()

df = df.reset_index(drop=True)

print("After removing duplicates:", len(df))
print("Remaining duplicate hashes:", df["resume_hash"].duplicated().sum())

Before removing duplicates: 4879
After removing duplicates: 4877
Remaining duplicate hashes: 0


In [54]:
# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: divide temporary data equally
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

Training records: 3901
Validation records: 488
Testing records: 488


In [61]:
def convert_to_prompt_completion(row):

    prompt_messages = [
        {
            "role": "system",
            "content": row["system"]
        },
        {
            "role": "user",
            "content": row["user"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = json.dumps(
        row["assistant"],
        ensure_ascii=False
    ) + tokenizer.eos_token

    return {
        "prompt": prompt,
        "completion": completion
    }

In [62]:
train_data = [
    convert_to_prompt_completion(row)
    for _, row in train_df.iterrows()
]

val_data = [
    convert_to_prompt_completion(row)
    for _, row in val_df.iterrows()
]

test_data = [
    convert_to_prompt_completion(row)
    for _, row in test_df.iterrows()
]

In [63]:
print("PROMPT:")
print(train_data[0]["prompt"])

print("\nCOMPLETION:")
print(train_data[0]["completion"])

PROMPT:
<|im_start|>system
You are an expert resume parser. Extract structured information from resumes and return ONLY valid JSON. Do not include explanations or extra text.<|im_end|>
<|im_start|>user
Resume:
LEAD ACCOUNTANT
Highlights
QuickBooks, Peachtree, In-house Accounting Systems and Financial Reporting, MAS 90, FAR, WAWF, Great Plains, Integration
Manager, FRx reporting.
Hyperion Workspace, Planning, Enterprise, Essbase, Schedules and Reports, Crystal Reports, MS Office with strong Excel skill
Financial Accounting
Database Management
Budget Administration
Performance Reporting
Tax Preparation and Compliance
Forecasting/Trend Analysis
Strategic Planning
Cash-Flow Analysis
Variance Analysis
Sarbanes-Oxley Research and Compliance
Experience
Lead Accountant
November 2011 to Current Company Name ï¼​ City , State
Experience in various areas including, but not limited to: operating budgets, financial analysis/planning, financial statements and reporting,
accounting policies and proced

In [64]:
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Testing examples:", len(test_data))

assert len(train_data) == len(train_df)
assert len(val_data) == len(val_df)
assert len(test_data) == len(test_df)

assert all(
    isinstance(item["prompt"], str)
    and isinstance(item["completion"], str)
    for item in train_data
)

print("Conversion successful!")

Training examples: 3901
Validation examples: 488
Testing examples: 488
Conversion successful!


In [ ]:

OUTPUT_DIR = Path("data/processed/sft_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Users\vishn\Downloads\data\processed\sft_dataset


In [70]:
def save_jsonl(data, file_path):

    with open(file_path, "w", encoding="utf-8") as file:

        for item in data:
            file.write(
                json.dumps(item, ensure_ascii=False) + "\n"
            )

    print(f"Saved {len(data)} records to {file_path}")

In [71]:
save_jsonl(
    train_data,
    OUTPUT_DIR / "train.jsonl"
)

save_jsonl(
    val_data,
    OUTPUT_DIR / "validation.jsonl"
)

save_jsonl(
    test_data,
    OUTPUT_DIR / "test.jsonl"
)

Saved 3901 records to data\processed\sft_dataset\train.jsonl
Saved 488 records to data\processed\sft_dataset\validation.jsonl
Saved 488 records to data\processed\sft_dataset\test.jsonl
